# sensIA — primeiro treino da nossa CNN

**Objetivo:** aprender a distinguir um par de frames com corte (`1`) de um par sem corte (`0`). Rede pequena, inicializada do zero, sem pesos AutoShot.

**Este notebook usa o ZIP `shot_pilot_v1.zip` que já foi preparado.** Ele respeita a divisão por vídeo existente; não cria outro split e não usa os 27 vídeos reservados para teste.

Recomendação: abrir no **Google Colab**, com GPU se disponível. Execute as células em ordem. Primeiro coloque o ZIP no seu Google Drive; não precisa enviar vídeos ao GitHub. A leitura dos vídeos pode levar mais tempo que o treino.

Ao final: exemplos dos pares, curvas, matriz de confusão, precisão/recall/F1, erros, modelo salvo e registro local no MLflow. Os resultados são de **validação do piloto**, não de teste independente nem do vídeo inteiro.


## 1. Preparar o Colab

Instalamos apenas o MLflow. O Colab normalmente já oferece TensorFlow, OpenCV, pandas, matplotlib e scikit-learn. Se o ambiente pedir reinicialização após instalar, reinicie antes de continuar.

Este notebook não altera o `requirements.txt` do projeto: ao final salva as versões efetivamente utilizadas.


In [ ]:
%pip -q install "mlflow>=3,<4"


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import shutil
import subprocess
import sys
import zipfile

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import sklearn
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
import mlflow

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("OpenCV:", cv2.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Indicar onde está o ZIP

**Altere somente `ZIP_PATH` para o caminho real do seu arquivo no Drive.** Exemplo: crie uma pasta `sensIA` no Drive e coloque o ZIP nela.

As demais escolhas são iniciais do experimento: imagens de 96×96, até 10 épocas, batch 32 e saída positiva a partir de 0,5. Esse limiar é da decisão **corte/sem corte**, não de estímulo ou saúde. Não estamos procurando o melhor hiperparâmetro ainda.


In [ ]:
ZIP_PATH = Path("/content/drive/MyDrive/sensIA/shot_pilot_v1.zip")  # AJUSTE AQUI
EXPECTED_ZIP_SHA256 = "620c30271977cf8cf20ae1da981404d80db53735ae665c6513c99452f5c4be8c"

SEED = 42
IMAGE_SIZE = 96
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
THRESHOLD = 0.5

tf.keras.utils.set_random_seed(SEED)
WORK_DIR = Path("/content/sensia_piloto") if IN_COLAB else Path.cwd() / "sensia_piloto_local"
DATA_DIR = WORK_DIR / "dataset" / EXPECTED_ZIP_SHA256[:12]
RUN_DIR = WORK_DIR / "resultados" / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert ZIP_PATH.is_file(), f"ZIP não encontrado. Corrija ZIP_PATH: {ZIP_PATH}"
print("Conferindo o ZIP; pode demorar no Drive...")
ZIP_SHA256 = file_sha256(ZIP_PATH)
assert ZIP_SHA256 == EXPECTED_ZIP_SHA256, "Hash diferente do pacote da Ana. Confira a versão antes de continuar."

DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as package:
    # Impede que um caminho no ZIP saia da pasta de extração.
    for name in package.namelist():
        assert (DATA_DIR / name).resolve().is_relative_to(DATA_DIR.resolve()), name
    package.extractall(DATA_DIR)
print("Pacote conferido e extraído em:", DATA_DIR)


## 3. Conferir os dados e os conjuntos

O `pairs.csv` é a lista de exemplos. A coluna `split` determina onde cada exemplo entra. A rede não descobre essa separação sozinha: nosso código a respeita.

Esperamos 43 vídeos de treino, 11 de validação e 1.664 pares no total. As conferências abaixo também impedem que um vídeo reservado apareça nos exemplos. Não altere os dados para forçar essas contagens se houver divergência: confira a versão do ZIP.


In [ ]:
pairs = pd.read_csv(DATA_DIR / "metadata/pairs.csv")
videos = pd.read_csv(DATA_DIR / "metadata/videos.csv")
dataset_summary = json.loads((DATA_DIR / "metadata/summary.json").read_text())
reference = pd.read_csv(DATA_DIR / "reference/official_test_ids.csv")

assert set(pairs["split"]) == {"train", "validation"}
assert set(pairs["label"]) == {0, 1}
assert (pairs["frame_b"] == pairs["frame_a"] + 1).all()
assert (pairs["frame_a"] >= 0).all()
assert not pairs.duplicated(["video_name", "frame_a", "frame_b"]).any()

train_names = set(pairs.loc[pairs["split"] == "train", "video_name"])
val_names = set(pairs.loc[pairs["split"] == "validation", "video_name"])
test_names = set(reference["video_name"])
assert train_names.isdisjoint(val_names), "Um vídeo apareceu em treino e validação."
assert (train_names | val_names).isdisjoint(test_names), "Vídeo do teste oficial encontrado nos pares."

selected = videos[videos["split"].isin(["train", "validation"])]
assert set(selected["video_name"]) == train_names | val_names
assert selected["sha256"].is_unique, "Há arquivos idênticos entre os vídeos selecionados."

print("Vídeos de treino:", len(train_names))
print("Vídeos de validação:", len(val_names))
display(pd.crosstab(pairs["split"], pairs["label"]).rename(columns={0: "sem_corte", 1: "corte"}))
display(pairs.head())
assert len(train_names) == 43 and len(val_names) == 11 and len(pairs) == 1664


## 4. Carregar os pares de imagens

Lemos cada vídeo uma vez em sequência. Só guardamos os frames necessários, já reduzidos, sem criar uma pasta de imagens. Preservamos a proporção e completamos o quadrado com bordas pretas.

Cada exemplo tem **96×96×6**: três canais RGB do frame A e três do frame B. Os pixels ficam em `uint8` para economizar memória; a normalização será feita dentro da rede. No piloto, a matriz dos pares ocupa cerca de 92 MB (decimais).


In [ ]:
def prepare_frame(frame, size):
    height, width = frame.shape[:2]
    scale = min(size / height, size / width)
    small = cv2.resize(frame, (max(1, round(width * scale)), max(1, round(height * scale))))
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    h, w = small.shape[:2]
    y, x = (size - h) // 2, (size - w) // 2
    canvas[y:y+h, x:x+w] = small
    return cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB)


def load_pairs(table, video_root, size, video_metadata):
    # A posição no array permanece igual ao índice da tabela.
    assert list(table.index) == list(range(len(table)))
    data = np.empty((len(table), size, size, 6), dtype=np.uint8)
    hashes = video_metadata.set_index("video_name")["sha256"].to_dict()
    groups = table.groupby("relative_path", sort=True)
    for number, (relative, rows) in enumerate(groups, 1):
        path = (video_root / relative).resolve()
        assert path.is_relative_to(video_root.resolve()) and path.is_file(), path
        assert file_sha256(path) == hashes[rows.iloc[0]["video_name"]], f"Vídeo modificado: {relative}"
        needed = set(rows["frame_a"]) | set(rows["frame_b"])
        frames = {}
        capture = cv2.VideoCapture(str(path))
        try:
            assert capture.isOpened(), f"Não foi possível abrir: {relative}"
            index = 0
            while index <= max(needed):
                ok, frame = capture.read()
                if not ok:
                    break
                if index in needed:
                    frames[index] = prepare_frame(frame, size)
                index += 1
        finally:
            capture.release()
        missing = needed - frames.keys()
        assert not missing, f"Frames não lidos em {relative}: {sorted(missing)[:10]}"
        for row in rows.itertuples():
            data[row.Index] = np.concatenate([frames[row.frame_a], frames[row.frame_b]], axis=-1)
        print(f"[{number}/{len(groups)}] {relative}")
    return data


X = load_pairs(pairs, DATA_DIR / "videos", IMAGE_SIZE, selected)
y = pairs["label"].to_numpy(dtype=np.int32)
train_mask = pairs["split"].eq("train").to_numpy()
val_mask = pairs["split"].eq("validation").to_numpy()
X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
val_rows = pairs.loc[val_mask].reset_index(drop=True)
print("Treino:", X_train.shape, "Validação:", X_val.shape)


## 5. Olhar exemplos antes de treinar

Confira se os pares fazem sentido. A primeira imagem é o frame A e a segunda é o B. Se os rótulos parecerem errados, investigue a preparação em vez de tentar corrigir isso com uma rede maior.


In [ ]:
example_rng = np.random.default_rng(SEED)
example_ids = []
for label in [0, 1]:
    candidates = np.flatnonzero(y_train == label)
    example_ids.extend(example_rng.choice(candidates, size=min(2, len(candidates)), replace=False))

fig, axes = plt.subplots(len(example_ids), 2, figsize=(7, 3 * len(example_ids)), squeeze=False)
for row, index in enumerate(example_ids):
    axes[row, 0].imshow(X_train[index, :, :, :3])
    axes[row, 1].imshow(X_train[index, :, :, 3:])
    axes[row, 0].set_title(f"Frame A | rótulo: {y_train[index]}")
    axes[row, 1].set_title("Frame B")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Construir a nossa CNN

Três camadas convolucionais aprendem padrões do par. As camadas de pooling reduzem o tamanho das representações; a camada final com sigmoid produz um escore entre 0 e 1 para corte. Não interpretar esse escore como probabilidade calibrada.

Esta é uma arquitetura inicial pequena, construída com Keras, **sem transferência de pesos**. O Dropout é uma tentativa de regularização, não uma garantia contra overfitting.


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 6)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(16, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid"),
], name="sensia_cnn_pares_v1")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ],
)
model.summary()


## 7. Treinar e registrar no MLflow

Treino atualiza os pesos. Validação acompanha exemplos de outros vídeos. O EarlyStopping observa a loss de validação, para após três épocas sem melhora e restaura os pesos da melhor época.

Começamos sem reponderar as classes para ter uma referência simples. Como há três negativos por positivo, uma rede que sempre responde “sem corte” pode alcançar 75% de acurácia. Por isso também veremos recall e F1 para cortes.

O MLflow é local a esta sessão, com SQLite e artefatos. Não há servidor compartilhado. As versões e os dados usados ficam registrados. Semente fixa ajuda a repetir o experimento, mas não garante igualdade numérica entre todo hardware/ambiente.

Para iniciar outro experimento do zero, execute novamente a célula de construção da CNN antes do treino. Rodar só `fit` novamente continuaria com os pesos já aprendidos.


In [ ]:
TRACK_DIR = WORK_DIR / "mlflow"
TRACK_DIR.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri("sqlite:///" + (TRACK_DIR / "mlflow.db").resolve().as_posix())
client = mlflow.tracking.MlflowClient()
experiment_name = "sensia_cnn_piloto"
if client.get_experiment_by_name(experiment_name) is None:
    client.create_experiment(experiment_name, artifact_location=(TRACK_DIR / "artifacts").resolve().as_uri())
mlflow.set_experiment(experiment_name)

config = {
    "notebook_version": "01_treino_piloto_cnn_v1",
    "seed": SEED, "image_size": IMAGE_SIZE, "input_channels": 6,
    "epochs_max": EPOCHS, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "threshold": THRESHOLD,
    "class_weight": "none", "pretrained_weights": "none",
    "zip_sha256": ZIP_SHA256,
    "pairs_sha256": file_sha256(DATA_DIR / "metadata/pairs.csv"),
    "train_pairs": len(y_train), "validation_pairs": len(y_val),
    "train_videos": len(train_names), "validation_videos": len(val_names),
}
environment = {
    "python": platform.python_version(), "tensorflow": tf.__version__,
    "opencv": cv2.__version__, "numpy": np.__version__,
    "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
    "mlflow": mlflow.__version__, "platform": platform.platform(),
    "gpu": [str(device) for device in tf.config.list_physical_devices("GPU")],
}
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
(RUN_DIR / "requirements_observed.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
)
(RUN_DIR / "model_architecture.json").write_text(model.to_json())

class LogEpochMetrics(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        for name, value in (logs or {}).items():
            mlflow.log_metric(name, float(value), step=epoch)

with mlflow.start_run(run_name="cnn_pares_v1") as run:
    RUN_ID = run.info.run_id
    mlflow.log_params(config)
    mlflow.log_dict(environment, "environment.json")
    mlflow.log_artifacts(str(DATA_DIR / "metadata"), artifact_path="dataset_metadata")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        shuffle=True,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
            LogEpochMetrics(),
        ],
        verbose=2,
    )
    history_df = pd.DataFrame(history.history)
    history_df.index.name = "epoch_zero_based"
    history_df.to_csv(RUN_DIR / "history.csv")
    model.save(RUN_DIR / "modelo_piloto.keras")
    mlflow.log_metric("epochs_executed", len(history_df))
    mlflow.log_artifacts(str(RUN_DIR), artifact_path="initial_training")

print("MLflow run_id:", RUN_ID)
print("Resultados:", RUN_DIR)


## 8. Visualizar as curvas

Observe a tendência da loss e o afastamento entre treino e validação. Loss de treino caindo com piora na validação pode indicar overfitting; não conclua isso a partir de uma única oscilação. A loss e a acurácia ajudam no diagnóstico, mas não substituem a avaliação dos cortes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs_run = np.arange(1, len(history_df) + 1)
for ax, metric, title in zip(axes, ["loss", "accuracy"], ["Loss", "Acurácia"]):
    ax.plot(epochs_run, history_df[metric], label="Treino")
    ax.plot(epochs_run, history_df["val_" + metric], label="Validação")
    ax.set(title=title, xlabel="Época")
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RUN_DIR / "curvas.png", dpi=150)
plt.show()


## 9. Métricas e matriz de confusão

- **Precisão para corte:** entre os pares apontados como corte, quantos tinham esse rótulo?
- **Recall para corte:** entre os pares rotulados como corte, quantos foram encontrados?
- **F1 para corte:** combina precisão e recall.

Comparamos a CNN com uma referência que sempre prevê “sem corte”. O limiar 0,5 permanece fixo neste primeiro ensaio. As métricas são por **par amostrado**: não são F1 de eventos no vídeo completo, nem desempenho em teste independente. A proporção artificial de negativos influencia a interpretação da precisão e acurácia.


In [ ]:
val_scores = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0).ravel()
val_pred = (val_scores >= THRESHOLD).astype(int)
baseline_pred = np.zeros_like(y_val)


def metrics_row(name, truth, prediction):
    return {
        "modelo": name,
        "accuracy": accuracy_score(truth, prediction),
        "precision_corte": precision_score(truth, prediction, zero_division=0),
        "recall_corte": recall_score(truth, prediction, zero_division=0),
        "f1_corte": f1_score(truth, prediction, zero_division=0),
    }

comparison = pd.DataFrame([
    metrics_row("Sempre sem corte", y_val, baseline_pred),
    metrics_row("Nossa CNN", y_val, val_pred),
])
display(comparison.round(4))
comparison.to_csv(RUN_DIR / "comparacao_validacao.csv", index=False)

report = classification_report(y_val, val_pred, labels=[0, 1],
    target_names=["Sem corte", "Corte"], output_dict=True, zero_division=0)
display(pd.DataFrame(report).T.round(4))
(RUN_DIR / "classification_report.json").write_text(json.dumps(report, indent=2))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_val, val_pred, labels=[0, 1],
    display_labels=["Sem corte", "Corte"], cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Validação — pares amostrados")
plt.tight_layout()
plt.savefig(RUN_DIR / "matriz_confusao.png", dpi=150)
plt.show()

predictions = val_rows.copy()
predictions["score_corte"] = val_scores
predictions["predicted_label"] = val_pred
predictions.to_csv(RUN_DIR / "predicoes_validacao.csv", index=False)


## 10. Olhar erros do modelo

Mostramos até quatro falsos positivos e quatro falsos negativos. Não há problema se o modelo inicial errar: essa é a evidência que orienta a próxima hipótese. Não altere rótulos apenas porque a rede discordou deles.


In [ ]:
false_positives = np.flatnonzero((y_val == 0) & (val_pred == 1))[:4]
false_negatives = np.flatnonzero((y_val == 1) & (val_pred == 0))[:4]
error_ids = np.concatenate([false_positives, false_negatives])

if len(error_ids) == 0:
    print("Nenhum erro nesta amostra de validação. Isso não comprova generalização.")
else:
    fig, axes = plt.subplots(len(error_ids), 2, figsize=(8, 3 * len(error_ids)), squeeze=False)
    for row, index in enumerate(error_ids):
        item = val_rows.iloc[index]
        kind = "Falso positivo" if y_val[index] == 0 else "Falso negativo"
        axes[row, 0].imshow(X_val[index, :, :, :3])
        axes[row, 1].imshow(X_val[index, :, :, 3:])
        axes[row, 0].set_title(f"{kind} | frame {item.frame_a}")
        axes[row, 1].set_title(f"frame {item.frame_b} | score {val_scores[index]:.2f}")
        axes[row, 0].set_ylabel(item.video_name, fontsize=8)
        for ax in axes[row]:
            ax.set_xticks([])
            ax.set_yticks([])
    plt.tight_layout()
    plt.savefig(RUN_DIR / "erros_validacao.png", dpi=120)
    plt.show()


## 11. Salvar a avaliação e copiar os resultados para o Drive

Execute esta célula antes de encerrar o Colab. Ela acrescenta a avaliação à mesma execução MLflow e copia um ZIP de resultados para a pasta do dataset no Drive. **Não inclui os vídeos.**

O pacote contém o modelo, tabelas, figuras, configurações e uma cópia do armazenamento local MLflow. O banco registra caminhos locais: não é um servidor compartilhado nem uma migração automática do MLflow para outra máquina. Modelo e relatórios podem ser utilizados separadamente. Não execute o teste oficial agora.


In [ ]:
final_metrics = comparison.loc[comparison["modelo"] == "Nossa CNN"].iloc[0].drop("modelo").astype(float).to_dict()
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metrics({"validation_final_" + k: float(v) for k, v in final_metrics.items()})
    mlflow.log_artifacts(str(RUN_DIR), artifact_path="final_results")

(RUN_DIR / "mlflow_run_id.txt").write_text(RUN_ID)
export_dir = WORK_DIR / ("export_" + RUN_DIR.name)
export_dir.mkdir(exist_ok=False)
shutil.copytree(RUN_DIR, export_dir / "resultados")
shutil.copytree(TRACK_DIR, export_dir / "mlflow_local")
archive = Path(shutil.make_archive(str(WORK_DIR / ("resultados_" + RUN_DIR.name)), "zip", export_dir))
destination = ZIP_PATH.parent / "resultados_sensia"
destination.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, destination / archive.name)
print("Resultados preservados em:", destination / archive.name)
print("SHA-256 dos resultados:", file_sha256(destination / archive.name))
print("Guarde o notebook com suas saídas. Nenhum resultado de teste oficial foi gerado.")


## 12. O que registrar na conclusão do primeiro ensaio

1. O notebook conseguiu ler os pares e treinar sem erro?
2. Qual foi o recall/F1 para cortes, comparado ao baseline?
3. Como se comportaram as curvas? Quais erros visuais apareceram?
4. Qual hipótese será testada no próximo experimento, mantendo a versão dos dados?

Não chamamos este resultado de “modelo aprovado” apenas por executar ou superar 75% de acurácia. O conjunto é pequeno, os rótulos não foram todos revisados visualmente e negativos próximos de transições foram evitados: isso torna o piloto mais simples que detectar todos os cortes de um vídeo novo. Serão necessárias inferência contínua, avaliação por eventos e validação no domínio infantil.

### Referências oficiais

- [TensorFlow: tutorial de CNN](https://www.tensorflow.org/tutorials/images/cnn) — referência dos componentes; a arquitetura e a entrada de seis canais são escolhas deste piloto.
- [Keras: EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping).
- [TensorFlow: set_random_seed](https://www.tensorflow.org/api_docs/python/tf/keras/utils/set_random_seed).
- [OpenCV: VideoCapture](https://docs.opencv.org/4.x/d8/dfe/classcv_1_1VideoCapture.html).
- [scikit-learn: classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html).
- [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/).
- [AutoShot / SHOT](https://github.com/wentaozhu/AutoShot).

As referências sustentam as ferramentas. Resolução, arquitetura, hiperparâmetros e critérios do piloto são decisões experimentais, não prescrições desses autores. No repositório, consultar `docs/METODOLOGIA_REDE_NEURAL.md` e `docs/PREPARACAO_E_REPASSE_SHOT.md`.
